# PARC2026 — 73 M3 A100 training smoke

Notebook 72d PASS後の **training smoke-only** 工程です。fresh Colab VMでも自己完結するよう、最初に3モデルの固定source/venvだけを再構築し、その後72dの実micro-batch/gradient-accumulationを使って同一training schedule seedでforward/reverse × equal-data/equal-wall × 3モデルを短時間だけ実行します。72a/72b/72c probeは再実行しません。OpenVLAの7B mergeもここでは行いません。1800秒benchmarkは開始しません。smoke orchestratorは72cで依存確認済みのOpenVLA Python 3.10 venv上で動かし、stdout/stderrをDriveへ永続化します。


In [ ]:
import os, subprocess
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
if not os.environ.get('HF_TOKEN'):
    try:
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not os.environ.get('HF_TOKEN'):
    raise RuntimeError('HF_TOKEN is required; add it to Colab Secrets as HF_TOKEN')
ROOT = Path('/content/parc2026')
ROOT.mkdir(parents=True, exist_ok=True)
REPO = ROOT / 'py_AI_m3_training_smoke'
PIN = 'a7ad0ac5c4ff5c7b21e904befebc0a9d4610ee0f'
URL = 'https://github.com/yu37330/py_AI.git'
if not (REPO / '.git').exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError(f'Existing non-Git directory: {REPO}')
    subprocess.run(['git', 'clone', '--no-checkout', URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', PIN], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', '--force', PIN], check=True)
got = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if got != PIN:
    raise RuntimeError(f'Repository pin mismatch: {got}')
print('73 smoke code:', got, flush=True)
env = os.environ.copy()
env['PY_AI_REPO'] = str(REPO)
env['PARC_ROOT'] = str(ROOT)
env['PARC_DRIVE_ROOT'] = '/content/drive/MyDrive/parc2026-cache'
env['PARC_M3_EXECUTE'] = '1'  # explicit smoke opt-in; benchmark mode remains disabled
base = Path(env['PARC_DRIVE_ROOT']) / 'model-benchmark-v1/m3-training-smoke-v1'
try:
    subprocess.run(['python', '-u', str(REPO / 'tools/colab/prepare_m3_training_runtimes.py')], cwd=str(REPO), env=env, check=True)
    smoke_python = ROOT / 'venv-openvla-oft-m3/bin/python'
    if not smoke_python.is_file():
        raise FileNotFoundError(smoke_python)
    subprocess.run([str(smoke_python), '-u', str(REPO / 'tools/colab/run_m3_training_smoke_logged.py')], cwd=str(REPO), env=env, check=True)
except (subprocess.CalledProcessError, FileNotFoundError):
    print('=== 73 DIAGNOSTICS ===', flush=True)
    status = base / 'runtime-setup/runtime_setup_status.json'
    if status.is_file():
        print(status.read_text(encoding='utf-8', errors='replace'), flush=True)
    setup_log = base / 'runtime-setup/runtime_setup.log'
    if setup_log.is_file():
        lines = setup_log.read_text(encoding='utf-8', errors='replace').splitlines()
        print('=== RUNTIME SETUP LOG TAIL ===', flush=True)
        print('\n'.join(lines[-120:]), flush=True)
    orch_status = base / 'orchestrator/orchestrator_status.json'
    if orch_status.is_file():
        print('=== ORCHESTRATOR STATUS ===', flush=True)
        print(orch_status.read_text(encoding='utf-8', errors='replace'), flush=True)
    orch_log = base / 'orchestrator/orchestrator.log'
    if orch_log.is_file():
        lines = orch_log.read_text(encoding='utf-8', errors='replace').splitlines()
        print('=== ORCHESTRATOR LOG TAIL ===', flush=True)
        print('\n'.join(lines[-160:]), flush=True)
    plans = sorted((base / 'plans').glob('*.json'), key=lambda p: p.stat().st_mtime, reverse=True) if (base / 'plans').is_dir() else []
    if plans:
        print('=== LATEST SMOKE PLAN ===', plans[0], flush=True)
        print(plans[0].read_text(encoding='utf-8', errors='replace'), flush=True)
    results = sorted(base.glob('**/train_result.json'), key=lambda p: p.stat().st_mtime, reverse=True)
    if results:
        print('=== LATEST TRAIN RESULT ===', results[0], flush=True)
        print(results[0].read_text(encoding='utf-8', errors='replace'), flush=True)
    raise
print('=== 73 COMPLETE ===', flush=True)
print('Training smoke only. Full 1800-second M3 benchmark has NOT started.', flush=True)
